# Tutorial 2: Phase analysis with tree search
Dara is equipped with a parallelized tree search algorithm to identify possible phases
present in a given XRD pattern.

In this tutorial, we will try to identify the phases in one experimental solid-state
reaction sample between `GeO2` and `ZnO`.

> You can download this tutorial project from [here](https://CederGroupHub.github.io/dara/_static/tutorial.zip).

In [ ]:
# %pip install ipywidgets nbformat

In [ ]:
from pathlib import Path

from dara import search_phases

In [ ]:
# pattern_path = "tutorial_data/GeO2-ZnO_700C_60min.xrdml"
pattern_path = r"D:\AMIT_WORK\projects\xrd_pipeline\new_data\OCX 1401026_exported.xy"

# three elements are present in the sample
chemical_system = "Li-Ni-Mn-Co-O-H"

In [ ]:
import re

def strip_xy_header(path: str) -> str:
    """Return a path to a header-stripped version of the .xy file.

    COMMANDER-exported .xy files have a metadata line at the top that
    dara cannot parse. This strips any lines that don't begin with a number
    and writes a clean copy next to the original (suffix _clean.xy).
    """
    p = Path(path)
    lines = p.read_text(encoding="utf-8").splitlines()
    data_lines = [l for l in lines if re.match(r"^\s*[\d.\-]", l)]
    clean_path = p.parent / (p.stem + "_clean.xy")
    clean_path.write_text("\n".join(data_lines), encoding="utf-8")
    return str(clean_path)

pattern_path = strip_xy_header(pattern_path)
print(f"Using cleaned pattern file: {pattern_path}")

## Step 1: Prepare reference phases

Dara pre-builds an index of all the unique and low-energy phases in ICSD and COD
databases. It also implements a method to download CIF structures from COD data server
so that there is no need to obtain the offline database.

Before every search, we will need to gather all the reference phases in the chemical
system for the search algorithm. Dara provides `ICSDDatabase` and `CODDatabase` to do
the filtering.

In this example, we will use `CODDatabase` to download all the phases in the chemical system of `Ge-O-Zn`.

In [ ]:
import time
import tempfile
import requests
from pathlib import Path
from dara.cif import Cif
from dara.structure_db import CODDatabase

COD_URL = "https://www.crystallography.net/cod/{cod_id}.cif"
_COD_SESSION = requests.Session()
_COD_SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (research use; contact: your@email.com)"
})

def _download_cod_windows_fix(cod_id: str) -> Cif:
    max_retries = 5
    for attempt in range(max_retries):
        try:
            # Rate-limit: base delay + increasing penalty on each retry
            time.sleep(0.5 + attempt * 1.5)
            url = COD_URL.format(cod_id=cod_id)
            response = _COD_SESSION.get(url, timeout=30)
            response.raise_for_status()
            with tempfile.NamedTemporaryFile(mode="w+b", suffix=".cif", delete=False) as f:
                temp_path = f.name
                f.write(response.content)
            try:
                cif = Cif.from_file(temp_path)
            finally:
                Path(temp_path).unlink(missing_ok=True)
            cif.filename = str(cod_id)
            return cif
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt  # exponential backoff: 1, 2, 4, 8 s
                print(f"  [{cod_id}] attempt {attempt+1} failed ({e}), retrying in {wait}s…")
                time.sleep(wait)
            else:
                print(f"  [{cod_id}] all {max_retries} attempts failed: {e}")
                raise

CODDatabase._download_cod = staticmethod(_download_cod_windows_fix)


In [ ]:
from dara.structure_db import CODDatabase

# The COD database contains methods to filter phases in the chemical system
cod_database = CODDatabase()

# gather reference phases and save them to a directory called "cifs"
all_icsd_ids = cod_database.get_cifs_by_chemsys(chemical_system, dest_dir="cifs")

Since we are using a pre-filterd database (i.e., the COD), the downloaded CIF files will automatically be named according to the
following convention:

```
{composition}_{spacegroup}_(cod|icsd_{id})-{e_hull}.cif
```
Where the `e_hull` is the energy above the convex hull in meV/atom, as determined from
the Materials Project database for the ground-state entry with matching composition and spacegroup.

## Step 2: Search for phases

After preparing the reference CIFs, we can start the phase search on a provided XRD
pattern.

In [ ]:
import ray

# Windows fix: stale Ray instances from a previous run cause PermissionError in workers.
# Always do a clean shutdown + restart before searching.
if ray.is_initialized():
    ray.shutdown()
ray.init(runtime_env={"working_dir": None})

cifs_dir = Path("cifs").resolve()

# Curated subset: R-3m layered oxides (sg 166) are the structural family of NMC811.
# Including common NMC secondary phases to allow multi-phase detection.
# keep_patterns = [
#     "_166_",      # R-3m layered oxides: LiNiO2, LiCoO2, NMC variants
#     "LiMn2O4",    # spinel (common secondary)
#     "NiO_",       # NiO (under-calcined secondary)
#     "Li2O_",      # Li2O (secondary)
#     "MnO2_",      # MnO2
#     "Co3O4_",     # cobalt oxide secondary
# ]

# Exclude non-physical / mislabeled phases that shadow real NMC structures in express mode.
# CoO3 in R-3m (COD 1525955) is a disordered cobalt oxide — not a realistic NMC secondary.
# It has fewer atomic constraints so it consistently "wins" the 166 group in express mode,
# preventing LiNiO2/NMC phases from ever being considered.
# exclude_patterns = [
#     # "CoO3_166_",
# ]

curated_cifs = [
    f for f in cifs_dir.glob("*.cif")
    # if any(p in f.name for p in keep_patterns)
    # and not any(p in f.name for p in exclude_patterns)
]
print(f"Curated set: {len(curated_cifs)} phases (from {len(list(cifs_dir.glob('*.cif')))} total)")
for f in sorted(curated_cifs):
    print(f"  {f.name}")

search_results = search_phases(
    pattern_path=pattern_path,
    phases=curated_cifs,
    wavelength="Cu",
    instrument_profile="Aeris-fds-Pixcel1d-Medipix3",
    express_mode=False,   # don't group 166 phases — let each NMC composition compete individually
)

## Step 3: Result analysis
The returned search result will be a list of `SearchResult` object.

In [ ]:
search_results

In [ ]:
for i in range(len(search_results)):
    print(f"Rwp of solution {i} = {search_results[i].refinement_result.lst_data.rwp} %")

Each `SearchResult` has a `.visualize()` method to visualize the refined pattern and
missing/extra peaks in the solution. If there are no missing or extra peaks, this option
will not appear.

In [ ]:
search_results[0].visualize()

You can also view all the alternative phases in one solution from `SearchResult.phases` attribute.

In [ ]:
print("Phases found in solution 0:")
for i, phases_ in enumerate(search_results[0].phases):
    print(f"    - Phase {i}: {[phase.path.name for phase in phases_]}")